In [1]:
%matplotlib notebook
import numpy as np
from pylab import *
import pynq
import os
import pyxrt
%config Completer.use_jedi = False
from craco_testing.pyxrtutil import *

In [2]:
xclbin='/data/craco/ban115/CRACO-42-ddgrid-axim-lut/pipeline_4cu_keith/gitlab-build/pipeline_4cu_keith/hw.xilinx_u280_xdma_201920_3/binary_container_1/binary_container_1.xclbin'
xclbin = 'pipeline_4cu_keith/hw.xilinx_u280_xdma_201920_3/binary_container_1/binary_container_1.xclbin'
xclbin = '/data/craco/ban115/CRACO-42-ddgrid-axim-lut/pipeline_4cu_keith/hw.xilinx_u280_xdma_201920_3/binary_container_1/binary_container_1.xclbin'

In [4]:
device = pyxrt.device(0)
xbin = pyxrt.xclbin(xclbin)
uuid = device.load_xclbin(xbin)
iplist = xbin.get_ips()
for ip in iplist:
    print(ip.get_name())

fdmt_tunable_c32:fdmt_tunable_c32_1
krnl_ddgrid_reader_4cu:krnl_ddgrid_reader_4cu_1
krnl_grid_4cu:krnl_grid_4cu_1
krnl_grid_4cu:krnl_grid_4cu_2
krnl_grid_4cu:krnl_grid_4cu_3
krnl_grid_4cu:krnl_grid_4cu_4
fft2d:fft2d_1
fft2d:fft2d_2
fft2d:fft2d_3
fft2d:fft2d_4
krnl_boxc_4cu:krnl_boxc_4cu_1
AXI-LITE-DDR0
AXI-LITE-DDR1
AXI-LITE-HBM0
AXI-LITE-HBM1
AXI-LITE-HBM2
AXI-LITE-HBM3
AXI-LITE-HBM4
AXI-LITE-HBM5
AXI-LITE-HBM6
AXI-LITE-HBM7
AXI-LITE-HBM8
AXI-LITE-HBM9
AXI-LITE-HBM10
AXI-LITE-HBM11
AXI-LITE-HBM12
AXI-LITE-HBM13
AXI-LITE-HBM14
AXI-LITE-HBM15
AXI-LITE-HBM16
AXI-LITE-HBM17
AXI-LITE-HBM18
AXI-LITE-HBM19
AXI-LITE-HBM20
AXI-LITE-HBM21
AXI-LITE-HBM22
AXI-LITE-HBM23
AXI-LITE-HBM24
AXI-LITE-HBM25
AXI-LITE-HBM26
AXI-LITE-HBM27
AXI-LITE-HBM28
AXI-LITE-HBM29
AXI-LITE-HBM30
AXI-LITE-HBM31


In [5]:
NDOUT = 186
NT = 256
NBLK = 3
NT_OUTBUF = NBLK*NT
NCIN = 32
NUV = 4800
NUVWIDE = 8
NUREST = int(NUV / NUVWIDE)
NDM_MAX = 32
NPIX = 256
   
class DdgridCu(Kernel):
    def __init__(self, device, xbin):
        super().__init__(device, xbin, 'krnl_ddgrid_reader_4cu:krnl_ddgrid_reader_4cu_1')
  
        
class FfftCu(Kernel):
    def __init__(self, device, xbin, icu):
        super().__init__(device, xbin, f'fft2d:fft2d_{icu+1}')
 
class GridCu(Kernel):
    def __init__(self, device, xbin, icu):
        super().__init__(device, xbin, f'krnl_grid_4cu:krnl_grid_4cu_{icu+1}')

        
class BoxcarCu(Kernel):
    def __init__(self, device, xbin):
        super().__init__(device, xbin, f'krnl_boxc_4cu:krnl_boxc_4cu_1')

class FdmtCu(Kernel):
    def __init__(self, device, xbin):
        super().__init__(device, xbin, 'fdmt_tunable_c32:fdmt_tunable_c32_1')
        
        

In [6]:
class Pipeline:
    def __init__(self, device, xbin):
        self.grid_reader = DdgridCu(device, xbin)
        self.grids = [GridCu(device, xbin, i) for i in range(4)]
        self.ffts = [FfftCu(device, xbin, i) for i in range(4)]
        self.boxcarcu = BoxcarCu(device, xbin)
        self.fdmtcu = FdmtCu(device, xbin)
        
        print('Allocating grid LUTs')
        self.grid_luts = [Buffer(8*1024*1024, np.uint32, device, g.group_id(3)).clear() for g in self.grids]
        
        # FDMT: (pin, pout, histin, histout, pconfig, out_tbkl)
        print('Allocating FDMT Input')

        self.inbuf = Buffer((NUV, NCIN, NT, 2), np.int16, device, self.fdmtcu.krnl.group_id(0)).clear()        
                
        # FDMT histin, histhout should be same buffer
        assert self.fdmtcu.group_id(2) == self.fdmtcu.group_id(3), 'FDMT histin and histout should be the same'
        
        print('Allocating FDMT history')
        self.fdmt_hist_buf = Buffer((256*1024*1024), np.int8, device, self.fdmtcu.krnl.group_id(2)).clear() # Grr, group_id puts you in some weird addrss space self.fdmtcu.krnl.group_id(2))
        
        print('Allocating FDMT fdmt_config_buf')
        self.fdmt_config_buf = Buffer((256*1024*1024), np.int8, device, self.fdmtcu.krnl.group_id(4)).clear()

        # pout of FDMT should be pin of grid reader
        assert self.fdmtcu.group_id(1) == self.grid_reader.group_id(0)

        # Grid reader: pin, ndm, tblk, nchunk, nparallel, axilut, load_luts, streams[4]
        print('Allocating mainbuf')
        self.mainbuf = Buffer((NUREST, NDOUT, NT_OUTBUF, NUVWIDE,2), np.int16, device, self.grid_reader.krnl.group_id(0)).clear()

        print('Allocating ddreader_lut')
        self.ddreader_lut = Buffer((NDM_MAX + NUREST), np.uint32, device, self.grid_reader.group_id(5)).clear()
        print('Allocating boxcar_history')    
        self.boxcar_history = Buffer((NDM_MAX, NPIX, NPIX, 2), np.int16, device, self.boxcarcu.group_id(3)).clear() # Grr, gruop_id problem self.boxcarcu.group_id(3))
        print('Allocating candidates')    
        self.candidates = Buffer(256*1024*1024, np.int8, device, self.boxcarcu.group_id(5)).clear() # Grrr self.boxcarcu.group_id(3))

        

In [7]:
p = Pipeline(device, xbin)

Kernel krnl_ddgrid_reader_4cu:krnl_ddgrid_reader_4cu_1 has groups
GID=0=32
GID=1=-1
GID=2=-1
GID=3=-1
GID=4=-1
GID=5=15
Kernel krnl_grid_4cu:krnl_grid_4cu_1 has groups
GID=0=-1
GID=1=-1
GID=2=-1
GID=3=6
Kernel krnl_grid_4cu:krnl_grid_4cu_2 has groups
GID=0=-1
GID=1=-1
GID=2=-1
GID=3=7
Kernel krnl_grid_4cu:krnl_grid_4cu_3 has groups
GID=0=-1
GID=1=-1
GID=2=-1
GID=3=20
Kernel krnl_grid_4cu:krnl_grid_4cu_4 has groups
GID=0=-1
GID=1=-1
GID=2=-1
GID=3=21
Kernel fft2d:fft2d_1 has groups
Kernel fft2d:fft2d_2 has groups
Kernel fft2d:fft2d_3 has groups
Kernel fft2d:fft2d_4 has groups
Kernel krnl_boxc_4cu:krnl_boxc_4cu_1 has groups
GID=0=-1
GID=1=-1
GID=2=-1
GID=3=54
GID=4=54
GID=5=4
Kernel fdmt_tunable_c32:fdmt_tunable_c32_1 has groups
GID=0=52
GID=1=32
GID=2=53
GID=3=53
GID=4=14
Allocating grid LUTs
Allocated 33554432 bytes flags=flags.normal groupid=6 address=0x60000000
Allocated 33554432 bytes flags=flags.normal groupid=7 address=0x70000000
Allocated 33554432 bytes flags=flags.normal groupid

In [7]:
lutbin = os.path.join(os.path.dirname(xclbin), '../../', 'none_duplicate_long.uvgrid.txt.bin')
lut = np.fromfile(lutbin, dtype=np.uint32)
for l in p.grid_luts:
    l.nparr[:len(lut)] = lut
    l.copy_to_device()


In [8]:
p.mainbuf.nparr[:] = 1
p.mainbuf.copy_to_device()
p.candidates.copy_from_device()
print(np.all(p.candidates.nparr == 0))

True


In [9]:
def run(self, threshold = 1.0):
    self = p
    ndm = 4
    nchunk_time = 32
    tblk = 0
    nuv = 3440
    nparallel_uv = nuv/2
    nurest = nuv/8
    load_luts = 1

    nplane = ndm*nchunk_time
    shift1 = 0 # FFT CONFIG register - not sure what this means
    shift2 = 7 # FFT CONFIG Register - not sure what this means
    fft_cfg = (nplane << 16) + (shift2 << 6) + (shift1 << 3)

    starts = []
    starts.append(self.grid_reader(self.mainbuf, ndm, tblk, nchunk_time, nurest, self.ddreader_lut, load_luts))
    starts.append(self.boxcarcu(ndm, nchunk_time, threshold, self.boxcar_history, self.boxcar_history, self.candidates))
    for cu in self.ffts:
        starts.append(cu(fft_cfg, fft_cfg))

    for cu, grid_lut in zip(self.grids, self.grid_luts):
        starts.append(cu(ndm, nchunk_time, nparallel_uv, grid_lut, load_luts))


    #for s in starts:
    #    starts.wait()
    print(starts)
    
    return starts

        
    

In [10]:
p.grid_reader.krnl.read_register(0x00)

4

In [11]:
starts = run(p)

print(starts)

[<pyxrt.run object at 0x7fcd434e3fb8>, <pyxrt.run object at 0x7fcd5c13aa40>, <pyxrt.run object at 0x7fcd5c13a4c8>, <pyxrt.run object at 0x7fcd5c13ad50>, <pyxrt.run object at 0x7fcd5c13a688>, <pyxrt.run object at 0x7fcd5c13a6c0>, <pyxrt.run object at 0x7fcd5c0d6f80>, <pyxrt.run object at 0x7fcd5c0d6df8>, <pyxrt.run object at 0x7fcd5c0d6d18>, <pyxrt.run object at 0x7fcd5c0d6e68>]
[<pyxrt.run object at 0x7fcd434e3fb8>, <pyxrt.run object at 0x7fcd5c13aa40>, <pyxrt.run object at 0x7fcd5c13a4c8>, <pyxrt.run object at 0x7fcd5c13ad50>, <pyxrt.run object at 0x7fcd5c13a688>, <pyxrt.run object at 0x7fcd5c13a6c0>, <pyxrt.run object at 0x7fcd5c0d6f80>, <pyxrt.run object at 0x7fcd5c0d6df8>, <pyxrt.run object at 0x7fcd5c0d6d18>, <pyxrt.run object at 0x7fcd5c0d6e68>]


In [ ]:
starts[2].wait(0)

In [ ]:
p.candidates.copy_from_device()
print(np.all(p.candidates.nparr == 0))
p.boxcar_history.copy_from_device()
print(np.all(p.boxcar_history.nparr == 0))

In [ ]:
p.inbuf.nparr[:] = 1
p.inbuf.copy_to_device()
p.fdmt_hist_buf.nparr[:] = 0
p.fdmt_hist_buf.copy_to_device()
print('inbuf', hex(p.inbuf.buf.address()))
print('mainbuf', hex(p.mainbuf.buf.address()))
print('histbuf', hex(p.fdmt_hist_buf.buf.address()))
print('fdmt_config_buf', hex(p.fdmt_config_buf.buf.address()))

In [ ]:
x = p.fdmtcu.krnl(p.inbuf.buf, p.mainbuf.buf, p.fdmt_hist_buf.buf, p.fdmt_hist_buf.buf, p.fdmt_config_buf.buf, 0)


In [ ]:

x.wait(100)

In [ ]:
f'{p.inbuf.buf.address}'

In [10]:
p.boxcar_history.buf.address

<bound method PyCapsule.address of <pyxrt.bo object at 0x7f9819142928>>

In [12]:
p.boxcar_history.nparr.flat[:10] = np.arange(10)


In [13]:
p.boxcar_history.nparr.flat[:10] 

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int16)

In [16]:
p.boxcar_history.copy_from_device()

In [17]:
p.boxcar_history.nparr.flat[:10] 

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int16)

In [18]:
p.boxcar_history.nparr.flat[:10] = np.arange(10)


In [25]:
buf  = Buffer(10, np.int8, device, 0).clear() # Grrr self.boxcarcu.group_id(3))


Allocated 10 bytes flags=flags.normal groupid=0 address=0x800000


In [26]:
print(buf.nparr)

[0 0 0 0 0 0 0 0 0 0]


In [28]:
buf.nparr[:] = np.arange(10)

In [29]:
buf.nparr

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int8)

In [30]:
buf.copy_to_device()

In [31]:
buf.nparr

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int8)

In [32]:
buf.nparr[:] = 0

In [33]:
buf.nparr

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8)

In [34]:
buf.copy_from_device()

In [36]:
buf.nparr

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int8)

In [37]:
device

In [38]:
mybo = pyxrt.bo(device, 10, pyxrt.bo.flags.device_only, 0)

In [40]:
mybo.address()


8392704

In [41]:
mybo_map = mybo.map()

RuntimeError: device only buffer has no host buffer: Invalid argument

In [43]:
mybo.sync(pyxrt.xclBOSyncDirection.XCL_BO_SYNC_BO_TO_DEVICE,  1, 0)

RuntimeError: unable to sync BO: Operation not supported

In [ ]:
device.